# GISLR — Stage 1 — phonology-only streaming models, trained in parallel (TODO §3.10)

**Pipeline stage (training), run by the user.** The user's requests (2026-09-26): train only on the
phonological features, only models that stream, as fast as the machine allows; then apply the follow-ups.

- **Input:** the 130 per-frame phonological features of `sb.recognize.phonology` (feature pipeline
  `phono130_v1`, all 543 landmarks in, no raw coordinates). Caches built (clips: key `157b4b1550f45acf`;
  GISLR-Sentences streams for the continuous eval).
- **Jobs** (all streaming), each its own process on the one GPU, started together:
  - isolated: `gru_phono130`, `lstm_phono130`, `cnn1d_phono130` (config `gislr.training.json`);
  - continuous: **`P1`** = `gru_continuous_phono130`, C1's sentence recipe on the phonology features, streams
    composed in feature space (`sb.recognize.continuous.phono`; config `gislr.continuous.json`).
- **`RETRAIN`**: jobs to run again although finished (the driver never reuses a finished run; it starts a new
  one). Default: `cnn1d_phono130`, whose first run stopped before its LR decayed (now standard patience 15).
- Hyperparameters live only in the two configs.

| artifact | path |
|---|---|
| run folders | `registry/runs/<run_id>/` |
| training logs | `data/temp/phono130_train_<job>.log` |
| exact ensemble table | `data/cache/gislr/ensembles/streaming_ensembles.csv` |

**Resumable:** re-running §2 restarts any unfinished job; each run resumes from its `last.pt`.

First round (2026-09-26): `gru_phono130` 0.7487, `lstm_phono130` 0.7359, `cnn1d_phono130` 0.7208;
exact ensembles of streaming models: 2 → 0.7912, 3 → 0.8048, 4 → 0.8075 (`phonology-models.md` §8).

## 1. Setup

In [ ]:
# ============================================================
# Imports, tunables, job list, which runs exist
# ============================================================
import subprocess
import sys
import time

import pandas as pd
from tqdm.auto import tqdm

from sb.core.paths import ROOT_DIR, TEMP_DIR
from sb.mlops import registry as R
from sb.recognize.config import load_config
from sb.recognize.continuous.train import run_dir_for as continuous_run_dir
from sb.recognize.data import subset_tag

JOBS = {  # name -> kind; all streaming
    "gru_phono130": "isolated", "lstm_phono130": "isolated", "cnn1d_phono130": "isolated",
    "P1": "continuous",
}
RETRAIN = ["cnn1d_phono130"]  # train again although finished (a new run folder)
POLL_S = 20                    # seconds between progress checks

CFG = load_config()
LOGS = {j: TEMP_DIR / f"phono130_train_{j}.log" for j in JOBS}
TEMP_DIR.mkdir(parents=True, exist_ok=True)


def run_dir(job):
    """The run folder the registry pointer names for this job (None before it starts)."""
    if JOBS[job] == "continuous":
        return continuous_run_dir(job)
    return R.pointer_run_dir(f"{CFG.dataset}_{job}_{subset_tag(CFG.subsets_for(job)[0], CFG.coords_for(job))}")


def finished(job):
    d = run_dir(job)
    return d is not None and (d / "meta.json").exists() and R.load_meta(d)["training"].get("finished", False)


for j in JOBS:
    print(f"{j:16s} {JOBS[j]:10s} run {run_dir(j)}  finished {finished(j)}" + ("  -> RETRAIN" if j in RETRAIN else ""))

## 2. Train every pending job at once

One process per job, started together, output to its log. The bar shows each run's epoch and validation
score (isolated: val accuracy; continuous: validation-stream segment accuracy).

In [ ]:
# ============================================================
# Parallel training: one process per job on the one GPU
# ============================================================
CALL = {"isolated": "from sb.recognize.train import train_from_config; train_from_config({!r})",
        "continuous": "from sb.recognize.continuous.train import train_continuous; train_continuous({!r})"}
procs, before = {}, {j: run_dir(j) for j in JOBS}
for j, kind in JOBS.items():
    if finished(j) and j not in RETRAIN:
        print(f"{j}: finished ({run_dir(j).name})")
        continue
    procs[j] = subprocess.Popen([sys.executable, "-c", CALL[kind].format(j)], cwd=ROOT_DIR,
                                stdout=open(LOGS[j], "w"), stderr=subprocess.STDOUT)
    print(f"{j}: started (pid {procs[j].pid}), log {LOGS[j]}")

bar = tqdm(desc="epochs (all jobs)", unit="ep")
while procs and any(p.poll() is None for p in procs.values()):
    time.sleep(POLL_S)
    state = {}
    for j in procs:
        d = run_dir(j)
        if d is None or (j in RETRAIN and d == before[j]) or not (d / "meta.json").exists():
            continue  # not started yet (a retrain shows once its new run folder exists)
        m = R.load_meta(d)
        state[j] = (m["training"]["epochs_trained"], m["metrics"]["train_val_acc"] or 0.0)
    bar.n = sum(e for e, _ in state.values())
    bar.set_postfix_str(" | ".join(f"{j} ep{e} {v:.3f}" for j, (e, v) in state.items()))
    bar.refresh()
bar.close()
for j, p in procs.items():
    print(f"{j}: exit code {p.returncode}" + ("" if p.returncode == 0 else f" -> see {LOGS[j]}"))

## 3. Canonical evaluation

`sb-evaluate` on every finished run: the canonical 18,896-clip split, per-class accuracy (a continuous model
is scored on isolated clips through its gloss head, as C1 was). P1's **sentence** score comes from
`gislr.3.streaming.continuous-eval.ipynb` (P1 is in its config; the phonology streams are cached), run after
this notebook.

In [ ]:
# ============================================================
# Score finished runs canonically; compare
# ============================================================
from sb.recognize.evaluate import evaluate_run

BASELINES = {"gru_phono_raw ME_134": 0.7632, "gru ME_132": 0.7517, "C1 (gru_continuous ME_132)": 0.7188}
rows = []
for j in JOBS:
    d = run_dir(j)
    if not finished(j):
        print(f"{j}: not finished -- run §2 again")
        continue
    m = R.load_meta(d)
    if m["metrics"].get("eval_status") != "canonical":
        evaluate_run(d, verbose=False)
        m = R.load_meta(d)
    rows.append({"job": j, "run": d.name, "params": m["n_params"], "epochs": m["training"]["epochs_trained"],
                 "canonical_acc": m["metrics"]["overall_accuracy"], "macro_acc": m["metrics"]["macro_accuracy"],
                 "wall_min": m["training"]["wall_time_min"]})
TABLE = pd.DataFrame(rows)
for name, acc in BASELINES.items():
    TABLE[f"vs {name}"] = TABLE.canonical_acc - acc
print(TABLE.round(4).to_string(index=False))

## 4. Exact ensembles of streaming models

Every combination of 1–4 streaming models, full probabilities averaged (`sb.recognize.ensemble`; each
member's probabilities computed once and cached). The phonology-only models join the raw-landmark and
phono+raw GRUs. Nothing is trained.

In [ ]:
# ============================================================
# Ensembles: all streaming members, 1-4 at a time
# ============================================================
from sb.core.paths import CACHE_DIR
from sb.recognize import ensemble as E

MEMBERS = {"gru_raw_ME132": 1789559734, "lstm_raw_ME126": 1790338279, "gru_phono_raw_ME134": 1790355555}
for j in ("gru_phono130", "lstm_phono130", "cnn1d_phono130"):
    if finished(j):
        MEMBERS[j] = int(run_dir(j).name)
ENS = E.evaluate(MEMBERS)
out = CACHE_DIR / "gislr" / "ensembles"
out.mkdir(parents=True, exist_ok=True)
ENS.to_csv(out / "streaming_ensembles.csv", index=False)
print(ENS.groupby("n").head(4).round(4).to_string(index=False))